In [ ]:
# LIBRERIAS

from astropy.table import Table
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pymc3 as pm
import pymc3_ext as pmx
import exoplanet as xo
import os
from corner import corner
import theano.tensor as tt
from tqdm.notebook import tqdm
from scipy.stats import median_abs_deviation
import pandas as pd
from astroquery.mast import Catalogs
from scipy.interpolate import UnivariateSpline
import pandas as pd

from pathlib import Path
from scipy.interpolate import interp1d

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import ScalarFormatter



# CATALOGOS

In [ ]:
# Catálogos

# Observables obtenidos de teff, spt con massage

# Catalogo de resultados massage, el _1 indica modelo de mist y el _2 indica modelo de somers

path_cat = "/home/luciam/Documents/starry/Resultados_Orion/catalogo_completo_mist_sommers.csv"

cat = pd.read_csv(path_cat)
print(f"datos catálogo mist-somers:{cat.columns}")

#Catálogo de resultados obtenidos de la curva de luz con tess

path_priors = "/home/luciam/Documents/starry/Resultados_Orion/priors_completosv1.csv"

priors = pd.read_csv(path_priors)
print(f"datos catálogo priors:{priors.columns}")

#Malla de tracks de mist

path_model_1 = "/home/luciam/Documents/starry/Resultados_Orion/datos_estrellas_massage_orion/modelos/MIST_V2.csv"

mist = pd.read_csv(path_model_1)
print(f"datos tracks mist:{mist.columns}")

# Mallan de tracks Somers 17% manchas
path_model_2 = "/home/luciam/Documents/starry/Resultados_Orion/datos_estrellas_massage_orion/modelos/f017track_ms.csv"

somers17 = pd.read_csv(path_model_2)
print(f"datos tracks somers:{somers17.columns}")

# ============================================================
# TICS EXCLUIDOS (de ambos modelos, MIST y Somers17)
# ============================================================

EXCLUDED_TICS = [11412481, 264459857, 264483228, 138829403]

cat = cat[~cat["TIC"].isin(EXCLUDED_TICS)].reset_index(drop=True)
priors = priors[~priors["TIC"].isin(EXCLUDED_TICS)].reset_index(drop=True)

print(f"\nTICs excluidos: {EXCLUDED_TICS}")
print(f"Filas en cat tras exclusión: {len(cat)}")
print(f"Filas en priors tras exclusión: {len(priors)}")

# ============================================================
# IDENTIFICAR EL SECTOR DE CADA FILA EN priors
# ============================================================
# priors repite el TIC porque cada fila es un sector distinto de TESS.
# NO promediamos entre sectores: cada sector se ajusta por separado
# (para poder ver si el tamaño inferido cambia entre épocas/sectores).
# Si el archivo ya trae una columna de sector la usamos; si no, se
# numera cada aparición del mismo TIC en el orden en que aparece.

_sector_candidates = [c for c in priors.columns if "sector" in c.lower()]

if _sector_candidates:
    SECTOR_COL = _sector_candidates[0]
else:
    priors = priors.sort_values("TIC").reset_index(drop=True)
    priors["sector_idx"] = priors.groupby("TIC").cumcount() + 1
    SECTOR_COL = "sector_idx"

print(f"\nColumna de sector usada: '{SECTOR_COL}'")
display(priors.groupby("TIC")[SECTOR_COL].count().rename("n_sectores"))


In [ ]:
#Constantes del modelo

T_sun = 5772.0  # K


# identificadores en cada catálogo


# VARIABLES DEL MODELO

In [ ]:
# ============================================================
# DATOS OBSERVACIONALES — MIST
# ============================================================
# (una fila por TIC, tomada del catálogo massage/MIST)

data_obs_mist = cat[[
    "TIC",
    "T_mean_1",
    "T_std_1",
    "lgL_mean_1",
    "lgL_std_1",
    "age_mean_1",
    "age_std_1",
    "mass_mean_1",
    "mass_std_1"
]].copy()


# ============================================================
# DATOS OBSERVACIONALES — SOMERS+17
# ============================================================

data_obs_somers = cat[[
    "TIC",
    "T_mean_2",
    "T_std_2",
    "lgL_mean_2",
    "lgL_std_2",
    "age_mean_2",
    "age_std_2",
    "mass_mean_2",
    "mass_std_2"
]].copy()


# ============================================================
# AGREGAR DATOS DE TESS — MIST
# ============================================================
# El merge con priors (que tiene una fila por sector) expande
# automáticamente cada TIC a una fila por sector — no hace falta
# promediar ni deduplicar.

data_obs_mist = data_obs_mist.merge(
    priors[[
        "TIC",
        SECTOR_COL,
        "Depth_primary",
        "Depth_primary_err"
    ]],
    on="TIC",
    how="inner"
)


# ============================================================
# AGREGAR DATOS DE TESS — SOMERS+17
# ============================================================

data_obs_somers = data_obs_somers.merge(
    priors[[
        "TIC",
        SECTOR_COL,
        "Depth_primary",
        "Depth_primary_err"
    ]],
    on="TIC",
    how="inner"
)

print(f"Sistemas (TIC x sector) — MIST: {len(data_obs_mist)}")
print(f"Sistemas (TIC x sector) — Somers17: {len(data_obs_somers)}")


# CALCULO DE RADIOS a partir de observables

In [ ]:
# ============================================================
# RADIO PRIMARIO — MIST
# ============================================================

data_obs_mist["R1_mean"] = (
    10**(data_obs_mist["lgL_mean_1"] / 2)
    * (T_sun / data_obs_mist["T_mean_1"])**2
)

data_obs_mist["R1_std"] = (
    data_obs_mist["R1_mean"]
    * np.sqrt(
        ((np.log(10) / 2) * data_obs_mist["lgL_std_1"])**2
        +
        (2 * data_obs_mist["T_std_1"]
         / data_obs_mist["T_mean_1"])**2
    )
)

# ============================================================
# RADIO SECUNDARIO — MIST
# ============================================================

data_obs_mist["R2_mean"] = (
    np.sqrt(1 - data_obs_mist["Depth_primary"])
    * data_obs_mist["R1_mean"]
)

data_obs_mist["R2_std"] = np.sqrt(
    (1 - data_obs_mist["Depth_primary"])
    * data_obs_mist["R1_std"]**2
    +
    (
        data_obs_mist["R1_mean"]
        * data_obs_mist["Depth_primary_err"]
        /
        (2 * np.sqrt(1 - data_obs_mist["Depth_primary"]))
    )**2
)

# ============================================================
# RADIO PRIMARIO — SOMERS+17
# ============================================================

data_obs_somers["R1_mean"] = (
    10**(data_obs_somers["lgL_mean_2"] / 2)
    * (T_sun / data_obs_somers["T_mean_2"])**2
)

data_obs_somers["R1_std"] = (
    data_obs_somers["R1_mean"]
    * np.sqrt(
        ((np.log(10) / 2) * data_obs_somers["lgL_std_2"])**2
        +
        (2 * data_obs_somers["T_std_2"]
         / data_obs_somers["T_mean_2"])**2
    )
)

# ============================================================
# RADIO SECUNDARIO — SOMERS+17
# ============================================================

data_obs_somers["R2_mean"] = (
    np.sqrt(1 - data_obs_somers["Depth_primary"])
    * data_obs_somers["R1_mean"]
)

data_obs_somers["R2_std"] = np.sqrt(
    (1 - data_obs_somers["Depth_primary"])
    * data_obs_somers["R1_std"]**2
    +
    (
        data_obs_somers["R1_mean"]
        * data_obs_somers["Depth_primary_err"]
        /
        (2 * np.sqrt(1 - data_obs_somers["Depth_primary"]))
    )**2
)


# ============================================================
# TICs A EXCLUIR
# ============================================================

tics_excluir = [
    11412481,
    264459857,
    264483228,
    138829403
]


# ============================================================
# FILTRAR DATOS OBSERVACIONALES
# ============================================================

data_obs_mist = data_obs_mist[
    ~data_obs_mist["TIC"].isin(tics_excluir)
].copy()

data_obs_somers = data_obs_somers[
    ~data_obs_somers["TIC"].isin(tics_excluir)
].copy()

display(
    data_obs_mist[
        [
            "TIC",
            "age_mean_1",
            "age_std_1",
            "R1_mean",
            "R1_std",
            "R2_mean",
            "R2_std"
        ]
    ].head()
)

display(
    data_obs_somers[
        [
            "TIC",
            "age_mean_2",
            "age_std_2",
            "R1_mean",
            "R1_std",
            "R2_mean",
            "R2_std"
        ]
    ].head()
)


In [ ]:

# Teoricos modelo mist
age_mist_teo = mist["star_age"] / 1e6 #convertida a millones de años
mass_mist_teo = mist['star_mass']
logL_mist_teo = mist['log_L'] 
logTeff_mist_teo = mist['log_Teff']
logR_mist_teo = mist['log_R']

# Teoricos modelo somers 17 
ageGyr_steo = somers17['Age(Gyr)']
ageMyr_steo = somers17['Age(Gyr)'] * 1000 #convertida a millones de años
logL_steo = somers17['log(L/Lsun)']
mass_steo = somers17['Mass']
logR_steo = somers17['log(R/Rsun)']


# DATOS TEORICOS DE LAS ISOCRONAS

In [ ]:
# vamos a construir un csv que contenga toda la información para el caso de somers y mist
#Suposicion clave: tenemos que los datos de la estrella primaria son los dominantes en el espectro
# por lo que el calculo inicial de massage nos da (R1,masa1,Teff1,Lum1), ademas con la curva de luz pudimos estimar la profundidad
#del tránsito de lo que obtenemos los datos de la estrella secundaria
# (Radio2, masa2, teff2, Lum2)

# Organizar los datos

In [ ]:
# ============================================================
# COMPROBAR UNIDADES DE EDAD
# ============================================================

print("Edad observacional MIST:")
print(data_obs_mist["age_mean_1"].describe())

print("\nEdad teórica MIST:")
print(mist["star_age"].describe())

print("\nEdad observacional Somers+17:")
print(data_obs_somers["age_mean_2"].describe())

print("\nEdad teórica Somers+17 [Myr]:")
print(ageMyr_steo.describe())

In [ ]:
# ============================================================
# LOG RADIO SECUNDARIO — MIST
# ============================================================

data_obs_mist["logR2_mean"] = np.log10(
    data_obs_mist["R2_mean"]
)

data_obs_mist["logR2_std"] = (
    data_obs_mist["R2_std"]
    /
    (data_obs_mist["R2_mean"] * np.log(10))
)

# ============================================================
# LOG RADIO SECUNDARIO — SOMERS+17
# ============================================================

data_obs_somers["logR2_mean"] = np.log10(
    data_obs_somers["R2_mean"]
)

data_obs_somers["logR2_std"] = (
    data_obs_somers["R2_std"]
    /
    (data_obs_somers["R2_mean"] * np.log(10))
)

# ============================================================
# EXCLUIR FILAS SIN EDAD PARA EL AJUSTE — SOMERS+17
# ============================================================

data_obs_somers_fit = data_obs_somers[
    data_obs_somers["age_mean_2"].notna() &
    data_obs_somers["age_std_2"].notna()
].copy()

print(f"Sistemas (TIC x sector) con edad válida — Somers17: {len(data_obs_somers_fit)}")


In [ ]:
# ============================================================
# VERIFICACIÓN DE LOS OBSERVABLES PARA EL AJUSTE
# ============================================================

display(
    data_obs_mist[
        [
            "TIC",
            SECTOR_COL,
            "age_mean_1",
            "age_std_1",
            "R2_mean",
            "R2_std",
            "logR2_mean",
            "logR2_std"
        ]
    ]
)

display(
    data_obs_somers_fit[
        [
            "TIC",
            SECTOR_COL,
            "age_mean_2",
            "age_std_2",
            "R2_mean",
            "R2_std",
            "logR2_mean",
            "logR2_std"
        ]
    ]
)


# AJUSTE POR ISÓCRONA A EDAD FIJA (interpolación en masa) — TODOS LOS TIC / SECTORES

In [ ]:
# ============================================================
# AJUSTE POR ISÓCRONA A EDAD FIJA — FUNCIONES GENÉRICAS
# ============================================================
# En vez de buscar el mínimo chi² sobre la malla discreta, para cada
# sistema:
#   1) Se construye la isocrona del modelo en la edad observada
#      (age_obs, que viene de la primaria) interpolando en edad DENTRO
#      de cada track de masa. No se usa ningún track cuya cobertura en
#      edad no incluya age_obs (no se extrapola).
#   2) Sobre esa isocrona se invierte la relación radio-masa para
#      encontrar M2 tal que R(M2) = R2_obs. Tampoco se extrapola: si
#      R2_obs cae fuera del rango de la isocrona, el sistema queda sin
#      ajuste para ese modelo (candidato a necesitar otra malla, p. ej.
#      BHAC15, más apropiada para masas muy bajas).
#   3) Se estima un error aproximado de M2 por diferencias finitas
#      (propagación lineal simple, NO Monte Carlo) variando age_obs y
#      logR_obs dentro de sus barras de error.

def build_isochrone_slice(teo_df, target_age, min_tracks=3):
    """Interpola cada track de masa a target_age (Myr) y arma la
    isocrona resultante. Descarta tracks que no cubran esa edad."""
    rows = []
    for m, grp in teo_df.groupby("mass"):
        grp = grp.sort_values("age_Myr")
        ages = grp["age_Myr"].to_numpy()
        if len(ages) < 2:
            continue
        if target_age < ages.min() or target_age > ages.max():
            continue  # este track no cubre esta edad: no extrapolamos
        row = {"mass": m}
        for col in ["log_R", "log_L", "log_Teff"]:
            row[col] = np.interp(target_age, ages, grp[col].to_numpy())
        rows.append(row)

    if len(rows) < min_tracks:
        return None

    return pd.DataFrame(rows).sort_values("mass").reset_index(drop=True)


def invert_mass_from_radius(iso, target_logR):
    """Sobre una isocrona a edad fija, invierte log_R -> mass para
    encontrar la masa cuyo radio coincide con target_logR. No
    extrapola fuera del rango de log_R cubierto por la isocrona."""
    if iso is None:
        return None

    iso_by_R = iso.sort_values("log_R")
    logR_sorted = iso_by_R["log_R"].to_numpy()
    mass_sorted_by_R = iso_by_R["mass"].to_numpy()

    if target_logR < logR_sorted.min() or target_logR > logR_sorted.max():
        return None  # fuera de cobertura: no extrapolamos

    M_best = np.interp(target_logR, logR_sorted, mass_sorted_by_R)

    iso_by_mass = iso.sort_values("mass")
    logL_best = np.interp(M_best, iso_by_mass["mass"], iso_by_mass["log_L"])
    logTeff_best = np.interp(M_best, iso_by_mass["mass"], iso_by_mass["log_Teff"])

    return M_best, logL_best, logTeff_best


def isochrone_fit(age_obs, age_err, logR_obs, logR_err, teo_df):
    """Ajuste de la secundaria por isocrona a edad fija + inversión en
    masa. Devuelve (dict_de_resultados, None) si tuvo éxito, o
    (None, motivo) si el sistema queda fuera de la cobertura del
    modelo (edad o radio)."""

    iso = build_isochrone_slice(teo_df, age_obs)
    if iso is None:
        return None, "edad fuera de la cobertura del grid"

    result = invert_mass_from_radius(iso, logR_obs)
    if result is None:
        return None, "radio observado fuera de la cobertura de la isocrona a esta edad"

    M_best, logL_best, logTeff_best = result

    # --- propagación simple de errores (diferencias finitas) ---
    M_err = np.nan
    try:
        iso_up_age = build_isochrone_slice(teo_df, age_obs + age_err)
        iso_dn_age = build_isochrone_slice(teo_df, age_obs - age_err)
        r_up_age = invert_mass_from_radius(iso_up_age, logR_obs)
        r_dn_age = invert_mass_from_radius(iso_dn_age, logR_obs)
        dM_age = (r_up_age[0] - r_dn_age[0]) / 2 if (r_up_age and r_dn_age) else 0.0

        r_up_R = invert_mass_from_radius(iso, logR_obs + logR_err)
        r_dn_R = invert_mass_from_radius(iso, logR_obs - logR_err)
        dM_R = (r_up_R[0] - r_dn_R[0]) / 2 if (r_up_R and r_dn_R) else 0.0

        M_err = float(np.hypot(dM_age, dM_R))
    except Exception:
        M_err = np.nan

    return {
        "age_best_Myr": age_obs,          # por construcción: coincide con la edad observada
        "M2_best": M_best,
        "M2_best_err": M_err,
        "R2_best": 10 ** logR_obs,         # por construcción: coincide con R2_obs (dentro de la malla)
        "logL2_best": logL_best,
        "logTeff2_best": logTeff_best,
        "T2_best": 10 ** logTeff_best,
    }, None


In [ ]:
# ============================================================
# AJUSTE MIST — TODOS LOS TIC / SECTORES (isocrona a edad fija)
# ============================================================

mist_teo_df = pd.DataFrame({
    "mass": mass_mist_teo.to_numpy(),
    "age_Myr": age_mist_teo.to_numpy(),
    "log_R": logR_mist_teo.to_numpy(),
    "log_L": logL_mist_teo.to_numpy(),
    "log_Teff": logTeff_mist_teo.to_numpy(),
})

mist_rows = []

for _, row in data_obs_mist.iterrows():
    fit, reason = isochrone_fit(
        row["age_mean_1"], row["age_std_1"],
        row["logR2_mean"], row["logR2_std"],
        mist_teo_df
    )

    if fit is None:
        print(f"[MIST] TIC {row['TIC']} sector {row[SECTOR_COL]}: {reason}")
        continue

    mist_rows.append({
        "TIC": row["TIC"],
        "sector": row[SECTOR_COL],
        "age_obs_Myr": row["age_mean_1"],
        "age_obs_err_Myr": row["age_std_1"],
        "M1_obs": row["mass_mean_1"],
        "M1_obs_err": row["mass_std_1"],
        "T1_obs": row["T_mean_1"],
        "T1_obs_err": row["T_std_1"],
        "logL1_obs": row["lgL_mean_1"],
        "logL1_obs_err": row["lgL_std_1"],
        "R1_obs": row["R1_mean"],
        "R1_obs_err": row["R1_std"],
        "R2_obs": row["R2_mean"],
        "R2_obs_err": row["R2_std"],
        **fit,
    })

results_mist_df = pd.DataFrame(mist_rows)
print(f"Ajustes MIST calculados: {len(results_mist_df)} sistemas (TIC x sector)")
display(results_mist_df)


In [ ]:
# ============================================================
# AJUSTE SOMERS17 — TODOS LOS TIC / SECTORES (isocrona a edad fija)
# ============================================================

somers_teo_df = pd.DataFrame({
    "mass": mass_steo.to_numpy(),
    "age_Myr": ageMyr_steo.to_numpy(),
    "log_R": logR_steo.to_numpy(),
    "log_L": logL_steo.to_numpy(),
    "log_Teff": somers17["log(Teff)"].to_numpy(),
})

somers_rows = []

for _, row in data_obs_somers_fit.iterrows():
    fit, reason = isochrone_fit(
        row["age_mean_2"], row["age_std_2"],
        row["logR2_mean"], row["logR2_std"],
        somers_teo_df
    )

    if fit is None:
        print(f"[Somers17] TIC {row['TIC']} sector {row[SECTOR_COL]}: {reason}")
        continue

    somers_rows.append({
        "TIC": row["TIC"],
        "sector": row[SECTOR_COL],
        "age_obs_Myr": row["age_mean_2"],
        "age_obs_err_Myr": row["age_std_2"],
        "M1_obs": row["mass_mean_2"],
        "M1_obs_err": row["mass_std_2"],
        "T1_obs": row["T_mean_2"],
        "T1_obs_err": row["T_std_2"],
        "logL1_obs": row["lgL_mean_2"],
        "logL1_obs_err": row["lgL_std_2"],
        "R1_obs": row["R1_mean"],
        "R1_obs_err": row["R1_std"],
        "R2_obs": row["R2_mean"],
        "R2_obs_err": row["R2_std"],
        **fit,
    })

results_somers_df = pd.DataFrame(somers_rows)
print(f"Ajustes Somers17 calculados: {len(results_somers_df)} sistemas (TIC x sector)")
display(results_somers_df)


# FIGURAS (Edad-Radio + Diagrama HR) PARA TODOS LOS TIC / SECTORES

In [ ]:
# ============================================================
# FUNCIÓN DE GRAFICADO — MIST (Edad-Radio + HR), UN SISTEMA
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.linewidth": 1.1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "legend.frameon": False,
    "mathtext.fontset": "cm",
})

CMAP = plt.cm.cividis

mist_plot = mist.copy()
mist_plot["age_Myr"] = mist_plot["star_age"] / 1e6
mist_plot["R"] = 10 ** mist_plot["log_R"]
_mist_mass_col = "initial_mass" if "initial_mass" in mist_plot.columns else "star_mass"
_mist_norm = Normalize(vmin=mist_plot["star_mass"].min(), vmax=mist_plot["star_mass"].max())


def get_track(df, target_mass, mass_col, tol_frac=0.03):
    """Traza evolutiva completa cuya masa inicial es más cercana a target_mass."""
    ref = df.sort_values("age_Myr").groupby(mass_col, as_index=False).first()
    closest_val = ref.loc[(ref[mass_col] - target_mass).abs().idxmin(), mass_col]
    track = df[np.isclose(df[mass_col], closest_val, rtol=tol_frac)]
    return track.sort_values("age_Myr"), closest_val


def plot_system_mist(fit_row, out_dir):
    tic, sector = fit_row["TIC"], fit_row["sector"]

    M1, T1 = fit_row["M1_obs"], fit_row["T1_obs"]
    logL1, logL1_err = fit_row["logL1_obs"], fit_row["logL1_obs_err"]
    R1, R1_err = fit_row["R1_obs"], fit_row["R1_obs_err"]
    R2, R2_err = fit_row["R2_obs"], fit_row["R2_obs_err"]
    age_obs, age_err = fit_row["age_obs_Myr"], fit_row["age_obs_err_Myr"]

    logTeff1 = np.log10(T1)
    logTeff1_err = fit_row["T1_obs_err"] / (T1 * np.log(10))

    age_best = fit_row["age_best_Myr"]
    M2 = fit_row["M2_best"]
    R2_best = fit_row["R2_best"]
    logL2 = fit_row["logL2_best"]
    logTeff2 = fit_row["logTeff2_best"]

    track1, M1_mist = get_track(mist_plot, M1, _mist_mass_col)
    track2, M2_mist = get_track(mist_plot, M2, _mist_mass_col)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.8), layout="constrained")

    ax1.plot(track1["age_Myr"], track1["R"], lw=2.2, color="black",
             label=fr"MIST $M={M1_mist:.2f}\,M_\odot$")
    ax1.plot(track2["age_Myr"], track2["R"], lw=2.2, color="tab:orange",
             label=fr"MIST $M={M2_mist:.2f}\,M_\odot$")

    ax1.errorbar(age_obs, R1, xerr=age_err, yerr=R1_err, fmt="o", markersize=9,
                 color="black", markeredgecolor="white", markeredgewidth=1.4,
                 elinewidth=1.2, capsize=3, alpha=0.9, zorder=10, label="Primaria")
    ax1.errorbar(age_obs, R2, xerr=age_err, yerr=R2_err, fmt="s", markersize=8,
                 color="tab:orange", markeredgecolor="white", markeredgewidth=1.4,
                 elinewidth=1.2, capsize=3, alpha=0.9, zorder=10, label="Secundaria")
    ax1.scatter(age_best, R2_best, marker="*", s=260, color="gold",
                edgecolor="black", linewidth=1.1, zorder=20, label="Mejor ajuste MIST")
    ax1.axvline(age_best, ls="--", color="gray", alpha=0.6, lw=1)

    ax1.set_xscale("log")
    ax1.xaxis.set_major_formatter(ScalarFormatter())
    ax1.set_xlabel("Edad [Myr]", fontsize=13)
    ax1.set_ylabel(r"$R\,[R_\odot]$", fontsize=13)
    ax1.set_title("Edad-Radio", fontsize=14)
    ax1.grid(alpha=0.2, which="both")
    ax1.legend(fontsize=9, loc="upper right")

    age_tol = max(age_best * 0.01, 0.05)
    iso = mist_plot[np.abs(mist_plot["age_Myr"] - age_best) < age_tol].copy()
    iso = iso.sort_values("star_mass")

    ax2.scatter(iso["log_Teff"], iso["log_L"], c=iso["star_mass"], cmap=CMAP,
                norm=_mist_norm, s=22, alpha=0.9, rasterized=True, zorder=5)
    ax2.errorbar(logTeff1, logL1, xerr=logTeff1_err, yerr=logL1_err, fmt="o",
                 markersize=11, color="black", markeredgecolor="white",
                 markeredgewidth=1.4, elinewidth=1.2, capsize=3, zorder=20,
                 label=fr"Primaria ($M={M1:.2f}\,M_\odot$)")
    ax2.errorbar(logTeff2, logL2, fmt="s", markersize=10,
                 color="tab:orange", markeredgecolor="white", markeredgewidth=1.4,
                 zorder=20, label=fr"Secundaria ($M={M2:.2f}\,M_\odot$)")
    ax2.scatter(logTeff2, logL2, marker="*", s=260, color="gold",
                edgecolor="black", linewidth=1.1, zorder=25)
    ax2.invert_xaxis()
    ax2.set_xlabel(r"$\log(T_{\rm eff}/{\rm K})$", fontsize=13)
    ax2.set_ylabel(r"$\log(L/L_\odot)$", fontsize=13)
    ax2.set_title(f"Diagrama HR — MIST ({age_best:.2f} Myr)", fontsize=14)
    ax2.grid(alpha=0.2, which="both")
    ax2.legend(fontsize=9, loc="lower left")

    sm = ScalarMappable(norm=_mist_norm, cmap=CMAP)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=[ax1, ax2], pad=0.02, shrink=0.9)
    cbar.set_label(r"Masa [$M_\odot$]", fontsize=12)

    fig.suptitle(f"TIC {tic} (sector {sector}) — Modelos evolutivos MIST", fontsize=16, y=1.03)

    out_path = out_dir / f"TIC_{tic}_sector_{sector}_MIST.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


mist_fig_dir = Path("figs_mist")
mist_fig_dir.mkdir(exist_ok=True)

for _, fit_row in results_mist_df.iterrows():
    plot_system_mist(fit_row, mist_fig_dir)

print(f"Figuras MIST guardadas en: {mist_fig_dir.resolve()}")


In [ ]:
# ============================================================
# FUNCIÓN DE GRAFICADO — SOMERS17 (Edad-Radio + HR), UN SISTEMA
# ============================================================

somers17_plot = somers17.copy()
somers17_plot["age_Myr"] = somers17_plot["Age(Gyr)"] * 1000
somers17_plot["R"] = 10 ** somers17_plot["log(R/Rsun)"]
_somers_mass_col = "Mass"
_somers_norm = Normalize(vmin=somers17_plot["Mass"].min(), vmax=somers17_plot["Mass"].max())


def plot_system_somers(fit_row, out_dir):
    tic, sector = fit_row["TIC"], fit_row["sector"]

    M1, T1 = fit_row["M1_obs"], fit_row["T1_obs"]
    logL1, logL1_err = fit_row["logL1_obs"], fit_row["logL1_obs_err"]
    R1, R1_err = fit_row["R1_obs"], fit_row["R1_obs_err"]
    R2, R2_err = fit_row["R2_obs"], fit_row["R2_obs_err"]
    age_obs, age_err = fit_row["age_obs_Myr"], fit_row["age_obs_err_Myr"]

    logTeff1 = np.log10(T1)
    logTeff1_err = fit_row["T1_obs_err"] / (T1 * np.log(10))

    age_best = fit_row["age_best_Myr"]
    M2 = fit_row["M2_best"]
    R2_best = fit_row["R2_best"]
    logL2 = fit_row["logL2_best"]
    logTeff2 = fit_row["logTeff2_best"]

    track1, M1_somers = get_track(somers17_plot, M1, _somers_mass_col)
    track2, M2_somers = get_track(somers17_plot, M2, _somers_mass_col)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.8), layout="constrained")

    ax1.plot(track1["age_Myr"], track1["R"], lw=2.2, color="black",
             label=fr"Somers17 $M={M1_somers:.2f}\,M_\odot$")
    ax1.plot(track2["age_Myr"], track2["R"], lw=2.2, color="tab:orange",
             label=fr"Somers17 $M={M2_somers:.2f}\,M_\odot$")

    ax1.errorbar(age_obs, R1, xerr=age_err, yerr=R1_err, fmt="o", markersize=9,
                 color="black", markeredgecolor="white", markeredgewidth=1.4,
                 elinewidth=1.2, capsize=3, alpha=0.9, zorder=10, label="Primaria")
    ax1.errorbar(age_obs, R2, xerr=age_err, yerr=R2_err, fmt="s", markersize=8,
                 color="tab:orange", markeredgecolor="white", markeredgewidth=1.4,
                 elinewidth=1.2, capsize=3, alpha=0.9, zorder=10, label="Secundaria")
    ax1.scatter(age_best, R2_best, marker="*", s=260, color="gold",
                edgecolor="black", linewidth=1.1, zorder=20, label="Mejor ajuste Somers17")
    ax1.axvline(age_best, ls="--", color="gray", alpha=0.6, lw=1)

    ax1.set_xscale("log")
    ax1.xaxis.set_major_formatter(ScalarFormatter())
    ax1.set_xlabel("Edad [Myr]", fontsize=13)
    ax1.set_ylabel(r"$R\,[R_\odot]$", fontsize=13)
    ax1.set_title("Edad-Radio", fontsize=14)
    ax1.grid(alpha=0.2, which="both")
    ax1.legend(fontsize=9, loc="upper right")

    age_tol = max(age_best * 0.01, 0.05)
    iso = somers17_plot[np.abs(somers17_plot["age_Myr"] - age_best) < age_tol].copy()
    iso = iso.sort_values("Mass")
    iso_logTeff = iso.apply(somers_logTeff, axis=1)

    ax2.scatter(iso_logTeff, iso["log(L/Lsun)"], c=iso["Mass"], cmap=CMAP,
                norm=_somers_norm, s=22, alpha=0.9, rasterized=True, zorder=5)
    ax2.errorbar(logTeff1, logL1, xerr=logTeff1_err, yerr=logL1_err, fmt="o",
                 markersize=11, color="black", markeredgecolor="white",
                 markeredgewidth=1.4, elinewidth=1.2, capsize=3, zorder=20,
                 label=fr"Primaria ($M={M1:.2f}\,M_\odot$)")
    ax2.errorbar(logTeff2, logL2, fmt="s", markersize=10,
                 color="tab:orange", markeredgecolor="white", markeredgewidth=1.4,
                 zorder=20, label=fr"Secundaria ($M={M2:.2f}\,M_\odot$)")
    ax2.scatter(logTeff2, logL2, marker="*", s=260, color="gold",
                edgecolor="black", linewidth=1.1, zorder=25)
    ax2.invert_xaxis()
    ax2.set_xlabel(r"$\log(T_{\rm eff}/{\rm K})$", fontsize=13)
    ax2.set_ylabel(r"$\log(L/L_\odot)$", fontsize=13)
    ax2.set_title(f"Diagrama HR — Somers17 ({age_best:.2f} Myr)", fontsize=14)
    ax2.grid(alpha=0.2, which="both")
    ax2.legend(fontsize=9, loc="lower left")

    sm = ScalarMappable(norm=_somers_norm, cmap=CMAP)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=[ax1, ax2], pad=0.02, shrink=0.9)
    cbar.set_label(r"Masa [$M_\odot$]", fontsize=12)

    fig.suptitle(f"TIC {tic} (sector {sector}) — Modelos evolutivos Somers17", fontsize=16, y=1.03)

    out_path = out_dir / f"TIC_{tic}_sector_{sector}_Somers17.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


somers_fig_dir = Path("figs_somers17")
somers_fig_dir.mkdir(exist_ok=True)

for _, fit_row in results_somers_df.iterrows():
    plot_system_somers(fit_row, somers_fig_dir)

print(f"Figuras Somers17 guardadas en: {somers_fig_dir.resolve()}")


# TABLA RESUMEN — MIST vs SOMERS17, TODOS LOS TIC / SECTORES

In [ ]:
# ============================================================
# TABLA RESUMEN COMBINADA — MIST vs SOMERS17
# ============================================================
# Una fila por TIC + sector; permite comparar el ajuste MIST y
# Somers17 del mismo sistema, y ver si R2/edad varían de un
# sector a otro (posibles variaciones de tamaño / errores externos).

summary_all = results_mist_df.merge(
    results_somers_df,
    on=["TIC", "sector"],
    how="outer",
    suffixes=("_mist", "_somers")
)

summary_all = summary_all.sort_values(["TIC", "sector"]).reset_index(drop=True)

display(summary_all)

summary_all.to_csv("resumen_mist_somers_por_tic_sector.csv", index=False)
print("Tabla guardada en resumen_mist_somers_por_tic_sector.csv")
